# PROYECTO FINAL
## 01. Extracción y limpieza

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Carga de datasets
metadata = pd.read_csv('../datos/metadata.csv')
electricity_raw = pd.read_csv('../datos/electricity.csv')
chilledwater_raw = pd.read_csv('../datos/chilledwater.csv')
weather_raw = pd.read_csv('../datos/weather.csv')

# Aspecto inicial de los datos de sensores
print("\nColumnas en electricity (primeras 3):", list(electricity_raw.columns[:3]))
print("Estructura original de electricity:", electricity_raw.shape)
print("Estructura original de chilledwater:", chilledwater_raw.shape)
print("Estructura original de weather:", weather_raw.shape)

# 2. Transformación de ancho a largo (MELT)
# Pasamos de tener 1.579 columnas a tener una estructura limpia de 3 columnas: Fecha, Edificio y Consumo
elec_long = pd.melt(electricity_raw, id_vars=['timestamp'], var_name='building_id', value_name='kwh')
elec_long['tipo_consumo'] = 'Electricidad General'

chilled_long = pd.melt(chilledwater_raw, id_vars=['timestamp'], var_name='building_id', value_name='kwh')
chilled_long['tipo_consumo'] = 'Agua refrigerada (AC)'

# 3. Unificar ambos contadores en una sola tabla
df_sensores = pd.concat([elec_long, chilled_long], axis=0, ignore_index=True)
df_sensores.shape


Columnas en electricity (primeras 3): ['timestamp', 'Panther_parking_Lorriane', 'Panther_lodging_Cora']
Estructura original de electricity: (17544, 1579)
Estructura original de chilledwater: (17544, 556)
Estructura original de weather: (331166, 10)


(37421352, 4)

In [3]:
df_sensores.head()

,timestamp,building_id,kwh,tipo_consumo
0,2016-01-01 00:00:00,Panther_parking_Lorriane,0.0,Electricidad General
1,2016-01-01 01:00:00,Panther_parking_Lorriane,0.0,Electricidad General
2,2016-01-01 02:00:00,Panther_parking_Lorriane,0.0,Electricidad General
3,2016-01-01 03:00:00,Panther_parking_Lorriane,0.0,Electricidad General
4,2016-01-01 04:00:00,Panther_parking_Lorriane,0.0,Electricidad General


In [4]:
# 4. Contar cuántos valores nulos reales tenemos en la columna de consumo
nulos_kwh = df_sensores['kwh'].isnull().sum()
porcentaje_nulos = (nulos_kwh / df_sensores.shape[0]) * 100

print("--- DIAGNÓSTICO DE CALIDAD ---")
print(f"Lecturas vacías (sensores estropeados): {nulos_kwh}")
print(f"Porcentaje de datos perdidos: {porcentaje_nulos:.2f}%")

# 5. Limpieza. Eliminar filas donde kwh sea nulo
# No se pueden inventar ni analizar el consumo de una hora que no se registró

df_sensores_limpio = df_sensores.dropna(subset=['kwh']).copy()
print(f"\nDimensiones tras eliminar nulos: {df_sensores_limpio.shape[0]} filas y {df_sensores_limpio.shape[1]} columnas")

--- DIAGNÓSTICO DE CALIDAD ---
Lecturas vacías (sensores estropeados): 1988607
Porcentaje de datos perdidos: 5.31%

Dimensiones tras eliminar nulos: 35432745 filas y 4 columnas


In [5]:
# 6. MERGE final con los Edificios
# Cruzar los datos usando la columna común 'building_id'
df_estudio = pd.merge(df_sensores_limpio, metadata, on='building_id', how='inner')

print(f"Merge completado con éxito.")
print(f"Dimensiones del dataset final de trabajo: {df_estudio.shape}")

print(df_estudio.columns.tolist())

Merge completado con éxito.
Dimensiones del dataset final de trabajo: (35432745, 35)
['timestamp', 'building_id', 'kwh', 'tipo_consumo', 'site_id', 'building_id_kaggle', 'site_id_kaggle', 'primaryspaceusage', 'sub_primaryspaceusage', 'sqm', 'sqft', 'lat', 'lng', 'timezone', 'electricity', 'hotwater', 'chilledwater', 'steam', 'water', 'irrigation', 'solar', 'gas', 'industry', 'subindustry', 'heatingtype', 'yearbuilt', 'date_opened', 'numberoffloors', 'occupants', 'energystarscore', 'eui', 'site_eui', 'source_eui', 'leed_level', 'rating']


In [7]:
# 1. Asegurar formatos correctos de fecha
print("Optimizando formatos de fecha...")
df_estudio['timestamp'] = pd.to_datetime(df_estudio['timestamp'])
weather_raw['timestamp'] = pd.to_datetime(weather_raw['timestamp'])

# SOLUCIÓN AL ERROR: Forzamos a que site_id sea texto plano (string) en ambas tablas
print("Normalizando identificadores de ubicación...")
df_estudio['site_id'] = df_estudio['site_id'].astype(str)
weather_raw['site_id'] = weather_raw['site_id'].astype(str)

# 2. EL MERGE CORRECTO: Unir usando timestamp Y site_id simultáneamente
print("Fusionando clima por ubicación y hora...")
df_estudio_final = pd.merge(df_estudio, weather_raw, on=['timestamp', 'site_id'], how='left')

print(f"¡Merge del clima completado con éxito!")
print(f"Dimensiones reales del dataset final: {df_estudio_final.shape}")

# 3. GUARDADO: Guardamos el archivo comprimido para el Notebook 02
print("\nGuardando el dataset optimizado (CSV comprimido)...")
df_estudio_final.to_csv('../datos/01_dataset_limpio_y_unificado.csv.gz', index=False, compression='gzip')

print("--- FASE 1 COMPLETADA CON ÉXITO ---")

Optimizando formatos de fecha...
Normalizando identificadores de ubicación...
Fusionando clima por ubicación y hora...
¡Merge del clima completado con éxito!
Dimensiones reales del dataset final: (35432745, 43)

Guardando el dataset optimizado (CSV comprimido)...
--- FASE 1 COMPLETADA CON ÉXITO ---


### MUESTRA, ARCHIVO MUY PESADO

In [8]:
# 1. Definir columnas clave
columnas_finales = [
    'timestamp', 'building_id', 'kwh', 'tipo_consumo', 
    'primaryspaceusage', 'sqm', 'yearbuilt', 'numberoffloors', 
    'airTemperature', 'leed_level'
]

# 2. Filtrar columnas útiles
df_filtrado_cols = df_estudio_final[columnas_finales].copy()

# 3. Extraer muestra aleatoria del 15% (Unos 5.3 millones de filas)
# random_state=42 asegura que si repites el código, siempre elija las mismas filas
df_muestra = df_filtrado_cols.sample(frac=0.15, random_state=42).copy()

print(f"Dimensiones de la nueva muestra optimizada: {df_muestra.shape}")

# 4. Guardar sustituyendo el archivo anterior
print("Guardando muestra final optimizada...")
df_muestra.to_csv('../datos/01_dataset_limpio_y_unificado.csv.gz', index=False, compression='gzip')
print("--- ¡MUESTRA LISTA Y GUARDADA! ---")

Dimensiones de la nueva muestra optimizada: (5314912, 10)
Guardando muestra final optimizada...
--- ¡MUESTRA LISTA Y GUARDADA! ---
